# ⚡ MeatVision AI — High-Speed GPU Model Training (Google Colab)

This notebook trains both **Meat Species Classification** (Beef, Chicken, Fish, Pork) and **Meat Freshness Detection** (Fresh, Half Fresh, Spoiled) using Google Colab T4 GPU acceleration.

### 🚀 Quick Start Instructions:
1. Click **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Select **T4 GPU**.
2. Execute Cell 1 to set up PyTorch & project files.
3. Execute Cell 2 & Cell 3 to train both models in **~3 minutes total**.
4. Download the resulting `species_model.pth` and `freshness_model.pth` checkpoints back into your local `models/` directory.

### Step 1: Check GPU & Setup Environment

In [5]:
import torch
import os
from pathlib import Path

print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.')


CUDA Available: False
⚠️ WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.


### Step 2: Upload or Clone Dataset & Codebase
*(Upload `MeatVisionAI.zip` or mount Google Drive if dataset is in Drive)*

In [6]:
# Step 2: Setup Project Code, Dataset & Mount Drive
import os, sys, zipfile, subprocess
from pathlib import Path

# 1. Reset working directory to /content first to prevent getcwd shell-init errors
try:
    os.chdir('/content')
except Exception:
    pass

# 2. Mount Google Drive (Optional)
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        try:
            drive.mount('/content/drive')
            print('✅ Google Drive mounted.')
        except Exception:
            print('Google Drive mount skipped.')
except ImportError:
    pass

# 3. Sync latest MeatVision AI codebase from GitHub
print('🔄 Syncing latest MeatVision AI codebase from GitHub...')
subprocess.run(['rm', '-rf', '/content/MeatVisionAI'], check=False)
subprocess.run(['git', 'clone', 'https://github.com/JoelJames889/MeatVisionAI.git', '/content/MeatVisionAI'], check=False)

project_root = Path('/content/MeatVisionAI')

# Detect training script directory (supports all path structures)
script_dir = None
for candidate in [
    project_root / 'scripts' / 'training',
    project_root / 'Scripts' / 'models',
    project_root / 'scripts' / 'models'
]:
    if (candidate / 'train_species.py').exists():
        script_dir = candidate
        break

if script_dir:
    print(f'✅ Found training scripts at: {script_dir}')
    os.chdir(project_root)
    sys.path.insert(0, str(script_dir))
    print(f'📂 Working directory set to: {os.getcwd()}')
else:
    raise RuntimeError('Could not locate train_species.py in cloned repository.')

# 4. Auto-extract Dataset if Dataset.zip is uploaded to /content/
dataset_zips = list(Path('/content').glob('*dataset*.zip')) + list(Path('/content').glob('*Dataset*.zip'))
for dz in dataset_zips:
    print(f'📦 Extracting Dataset zip: {dz.name}...')
    with zipfile.ZipFile(dz, 'r') as z:
        z.extractall(project_root)

# 5. Verify Dataset presence
species_found = False
check_paths = [
    project_root / 'Dataset' / 'species',
    project_root / 'Dataset' / 'Species',
    project_root / 'Clean_Dataset' / 'Species',
    Path('/content/drive/MyDrive/Dataset/species'),
    Path('/content/drive/MyDrive/MeatVisionAI/Dataset/species'),
    Path('/content/Dataset/species')
]

for cp in check_paths:
    if cp.exists() and any(cp.iterdir()):
        species_found = True
        print(f'✅ Dataset verified at: {cp}')
        break

if not species_found:
    print('\n' + '='*80)
    print('⚠️ NOTICE: DATASET IMAGES NEEDED IN COLAB')
    print('='*80)
    print('To run training, please ensure dataset images are available:')
    print('  Option A: Mount Google Drive if dataset is in MyDrive/Dataset/')
    print('  Option B: Upload Dataset.zip to /content/ and run: !unzip /content/Dataset.zip -d /content/MeatVisionAI/')
    print('='*80 + '\n')


### Step 3: Train Species Classification Model (`species_model.pth`)

In [1]:
# Execute Species Training Script
!python scripts/training/train_species.py


python: can't open file 'd:\\MeatVision_Project\\notebooks\\scripts\\training\\train_species.py': [Errno 2] No such file or directory


### Step 4: Train Freshness Detection Model (`freshness_model.pth`)

In [ ]:
# Execute Freshness Training Script
!python scripts/training/train_freshness.py


### Step 5: Download Trained Checkpoints to Your Machine

In [ ]:
from google.colab import files
import os

models_to_download = ['models/species_model.pth', 'models/freshness_model.pth']
for m in models_to_download:
    if os.path.exists(m):
        print(f'Downloading {m}...')
        files.download(m)
    else:
        print(f'Checkpoint {m} not found.')
